In [ ]:
# 1. Workflow
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def box(ax, xy, text, w=2.6, h=0.82, color="#e8eef6"):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
        facecolor=color, edgecolor="#334155", linewidth=1.2, zorder=2,
    ))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8.1, color="#0f172a", zorder=3)
    return {"x": x, "y": y, "w": w, "h": h, "cx": x + w / 2, "cy": y + h / 2}


def edge_point(src, dst, pad=0.08):
    dx, dy = dst["cx"] - src["cx"], dst["cy"] - src["cy"]
    hw, hh = src["w"] / 2 + pad, src["h"] / 2 + pad
    tx = hw / abs(dx) if dx else float("inf")
    ty = hh / abs(dy) if dy else float("inf")
    t = min(tx, ty)
    return src["cx"] + t * dx, src["cy"] + t * dy


def arrow(ax, src, dst):
    ax.add_patch(FancyArrowPatch(
        edge_point(src, dst), edge_point(dst, src),
        arrowstyle="-|>", mutation_scale=9, linewidth=1.1,
        color="#475569", shrinkA=0, shrinkB=2, zorder=1,
    ))


fig, ax = plt.subplots(figsize=(12.4, 5.1))
ax.set_xlim(0, 12.6)
ax.set_ylim(0, 5.1)
ax.axis("off")
ax.set_title("CPU pipeline: VideoMAE as join key → 4-IMU FAME → Viterbi", loc="left", fontsize=12, pad=8)
c1 = box(ax, (0.15, 3.55), "Train 4 IMU\nFAME 400-d + mask", color="#dbeafe")
c2 = box(ax, (0.15, 1.75), "Test 1 IMU + 15×768\n(same camera, 4 files)", color="#dbeafe")
c3 = box(ax, (3.20, 2.65), "Sensor dropout\nLightGBM 19-way", color="#fef3c7")
c4 = box(ax, (6.20, 3.55), "Match video\nsame sbj, other limb", color="#dcfce7")
c5 = box(ax, (6.20, 1.75), "Rebuild (50,4,3)\n1–4 limbs present", color="#dcfce7")
c6 = box(ax, (9.35, 2.65), "Viterbi / subject\nstay=0.93, broadcast", color="#ede9fe")
arrow(ax, c1, c3)
arrow(ax, c2, c4)
arrow(ax, c3, c5)
arrow(ax, c4, c5)
arrow(ax, c5, c6)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Setup — V8_4IMU_VIDEO_JOIN_VITERBI
import gc
import os
import random
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)

PIPELINE = "V8_4IMU_VIDEO_JOIN_VITERBI"
SEED = 42
MODE = "full"  # "smoke" for a wiring check

print("=" * 72)
print(f"PIPELINE={PIPELINE}   MODE={MODE}")
print("If you do not see this banner in the logs, the old 0.639 notebook is still running.")
print("=" * 72)

try:
    import lightgbm as lgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb

CLASS_NAMES = [
    "null",
    "jogging", "jogging (rotating arms)", "jogging (skipping)",
    "jogging (sidesteps)", "jogging (butt-kicks)",
    "stretching (triceps)", "stretching (lunging)", "stretching (shoulders)",
    "stretching (hamstrings)", "stretching (lumbar rotation)",
    "push-ups", "push-ups (complex)",
    "sit-ups", "sit-ups (complex)",
    "burpees",
    "lunges", "lunges (complex)",
    "bench-dips",
]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
N_CLASSES = len(CLASS_NAMES)
SENSOR_LOCATIONS = ["right_arm", "right_leg", "left_leg", "left_arm"]
SENSOR_TO_ID = {name: i for i, name in enumerate(SENSOR_LOCATIONS)}
N_SENSORS = 4
INERTIAL_COLS = [f"{loc}_acc_{ax}" for loc in SENSOR_LOCATIONS for ax in "xyz"]

INERTIAL_HZ = 50
WINDOW_SAMPLES = 50
VIDEO_WINDOW = 15
VIDEO_DIM = 768
VIDEO_FRAME_OFFSET = 8
STRIDE = 25
MAX_PER_SUBJECT_CLASS = 800
N_ESTIMATORS = 700
LR = 0.07
NUM_LEAVES = 63
N_DROPOUT_COPIES = 2
N_SEEDS = 2
STAY_PROB = 0.93
NULL_BIAS = 0.35
MATCH_THRESHOLDS = (0.9995, 0.999, 0.997, 0.995, 0.99, 0.98)

if MODE == "smoke":
    STRIDE = 50
    MAX_PER_SUBJECT_CLASS = 30
    N_ESTIMATORS = 40
    N_DROPOUT_COPIES = 1
    N_SEEDS = 1

COMPETITION_NAME = "3rd-wear-dataset-challenge-hasca-2026"
CANDIDATE_ROOTS = [
    Path("/kaggle/input/competitions") / COMPETITION_NAME,
    Path("/kaggle/input") / COMPETITION_NAME,
    Path("data"),
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
TRAIN_INERTIAL_DIR = DATA_ROOT / "train" / "inertial_feat"
TRAIN_VIDEO_DIR = DATA_ROOT / "train" / "videomae_feat"
if not TRAIN_INERTIAL_DIR.exists():
    TRAIN_INERTIAL_DIR = DATA_ROOT / "train"
    TRAIN_VIDEO_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
TEST_INERTIAL_PATH = TEST_DIR / "test_inertial_data.npy"
TEST_VIDEO_PATH = TEST_DIR / "test_videomae_data.npy"
TEST_META_PATH = TEST_DIR / "test_meta_data.csv"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print(f"DATA_ROOT={DATA_ROOT} exists={DATA_ROOT.exists()}")


In [ ]:
# 3. FAME IMU features, windows, LightGBM, Viterbi

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def sbj_id_from_stem(stem: str) -> int:
    m = re.match(r"sbj_(\d+)", stem)
    return int(m.group(1)) if m else -1


def label_ids(series: pd.Series) -> np.ndarray:
    out = np.zeros(len(series), dtype=np.int64)
    for i, name in enumerate(series.to_numpy()):
        out[i] = LABEL_TO_ID[name] if isinstance(name, str) and name in LABEL_TO_ID else 0
    return out


def load_inertial_csv(path: Path):
    frame = pd.read_csv(path, low_memory=False, dtype={"label": "string"})
    missing = [c for c in INERTIAL_COLS if c not in frame.columns]
    if missing:
        raise KeyError(f"{path.name} missing {missing[:6]}")
    inertial = frame[INERTIAL_COLS].to_numpy(np.float32).reshape(len(frame), N_SENSORS, 3)
    labels = label_ids(frame["label"])
    sbj = int(frame["sbj_id"].iloc[0]) if "sbj_id" in frame.columns else sbj_id_from_stem(path.stem)
    return np.nan_to_num(inertial, nan=0.0), labels, sbj


def as_video_tn(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 2 and array.shape[-1] == VIDEO_DIM:
        return array
    if array.ndim == 2 and array.shape[0] == VIDEO_DIM:
        return array.T
    raise ValueError(f"Unexpected video shape {array.shape}")


def segment_starts(labels: np.ndarray, stride: int):
    labels = np.asarray(labels)
    cuts = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    seg_s = np.concatenate([[0], cuts])
    seg_e = np.concatenate([cuts, [len(labels)]])
    starts, targets = [], []
    for s, e in zip(seg_s, seg_e):
        if e - s < WINDOW_SAMPLES:
            continue
        first = ((s + stride - 1) // stride) * stride
        if first + WINDOW_SAMPLES > e:
            continue
        idx = np.arange(first, e - WINDOW_SAMPLES + 1, stride, dtype=np.int64)
        starts.append(idx)
        targets.append(np.full(len(idx), labels[s], dtype=np.int64))
    if not starts:
        return np.empty(0, np.int64), np.empty(0, np.int64)
    return np.concatenate(starts), np.concatenate(targets)


def subsample_subject_class(frame: pd.DataFrame, cap: int, seed: int) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    out = frame.copy()
    out["_r"] = rng.rand(len(out))
    out = out.sort_values("_r")
    out["_n"] = out.groupby(["sbj_id", "target"])["_r"].cumcount()
    return out[out["_n"] < cap].drop(columns=["_r", "_n"]).reset_index(drop=True)


def time_features(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = x.shape[-1]
    mean = x.mean(axis=-1)
    std = np.sqrt(np.mean((x - mean[..., None]) ** 2, axis=-1) + 1e-12)
    rms = np.sqrt(np.mean(x ** 2, axis=-1) + 1e-12)
    xmin, xmax = x.min(axis=-1), x.max(axis=-1)
    median = np.median(x, axis=-1)
    zm = x - mean[..., None]
    z2 = np.mean(zm ** 2, axis=-1) + 1e-12
    skew = np.mean(zm ** 3, axis=-1) / (z2 ** 1.5)
    kurt = np.mean(zm ** 4, axis=-1) / (z2 ** 2) - 3.0
    zc = (np.sign(zm[..., 1:]) * np.sign(zm[..., :-1]) < 0).sum(axis=-1) / max(1, n - 1)
    mcr = (np.sign(x[..., 1:] - mean[..., None]) * np.sign(x[..., :-1] - mean[..., None]) < 0).sum(axis=-1) / max(1, n - 1)
    ar1 = np.mean(zm[..., :-1] * zm[..., 1:], axis=-1) / (np.mean(zm[..., :-1] ** 2, axis=-1) + 1e-12)
    jerk = np.std(np.diff(x, axis=-1), axis=-1)
    out = np.stack([mean, std, rms, xmin, xmax, xmax - xmin, median, skew, kurt, zc, mcr, ar1, jerk], axis=-1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def freq_features(x: np.ndarray, fs: float = 50.0) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean(axis=-1, keepdims=True)
    mag = np.abs(np.fft.rfft(x, axis=-1))
    freqs = np.fft.rfftfreq(x.shape[-1], 1.0 / fs)
    power = mag ** 2
    total = np.maximum(power.sum(axis=-1, keepdims=True), 1e-12)
    p = power / total
    dom = freqs[power.argmax(axis=-1)]
    centroid = (power * freqs).sum(axis=-1) / total[..., 0]
    entropy = -(p * np.log(p + 1e-12)).sum(axis=-1)
    low = power[..., freqs < 2].sum(axis=-1) / total[..., 0]
    mid = power[..., (freqs >= 2) & (freqs < 8)].sum(axis=-1) / total[..., 0]
    high = power[..., freqs >= 8].sum(axis=-1) / total[..., 0]
    rolloff = freqs[np.argmax(np.cumsum(p, axis=-1) >= 0.95, axis=-1)]
    flux = np.mean(np.abs(np.diff(p, axis=-1)), axis=-1)
    return np.stack([dom, centroid, entropy, low, mid, high, rolloff, flux], axis=-1)


def subwindow_features(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = x.shape[-1]
    half = max(1, n // 2)
    x0, x1 = x[..., :half], x[..., half:2 * half]
    e0, e1 = np.mean(x0 ** 2, axis=-1), np.mean(x1 ** 2, axis=-1)
    t = np.linspace(0.0, 1.0, n)
    denom = np.sum((t - t.mean()) ** 2)
    slope = np.sum((x - x.mean(axis=-1, keepdims=True)) * (t - t.mean()), axis=-1) / denom
    return np.stack([
        x1.mean(axis=-1) - x0.mean(axis=-1),
        x1.std(axis=-1) - x0.std(axis=-1),
        e1 / (e0 + 1e-9),
        slope,
    ], axis=-1)


def axis_block(x: np.ndarray) -> np.ndarray:
    return np.concatenate([time_features(x), freq_features(x), subwindow_features(x)], axis=-1)


def imu_100(xyz: np.ndarray) -> np.ndarray:
    """(N, 50, 3) -> (N, 100). Year-2 FAME block, one sensor."""
    xyz = np.nan_to_num(np.asarray(xyz, dtype=np.float64), nan=0.0)
    parts = [axis_block(xyz[..., a]) for a in range(3)]
    parts.append(axis_block(np.linalg.norm(xyz, axis=-1)))
    return np.concatenate(parts, axis=-1)


def mag_corr_4(imu: np.ndarray, present: np.ndarray) -> np.ndarray:
    """Pairwise magnitude correlations. imu (N,50,4,3), present (N,4) -> (N,6)."""
    mag = np.linalg.norm(np.asarray(imu, dtype=np.float64), axis=-1)
    mag = mag - mag.mean(axis=1, keepdims=True)
    cols = []
    k = 0
    out = np.zeros((len(imu), 6), dtype=np.float64)
    for i in range(N_SENSORS):
        for j in range(i + 1, N_SENSORS):
            num = (mag[:, :, i] * mag[:, :, j]).sum(axis=1)
            den = np.sqrt((mag[:, :, i] ** 2).sum(axis=1) * (mag[:, :, j] ** 2).sum(axis=1)) + 1e-9
            ok = present[:, i] * present[:, j]
            out[:, k] = (num / den) * ok
            k += 1
    return out


def features_4(imu: np.ndarray, present: np.ndarray) -> np.ndarray:
    """imu (N,50,4,3), present (N,4) -> (N, 410). Missing limbs zeroed."""
    present = present.astype(np.float64)
    cols = []
    for s in range(N_SENSORS):
        cols.append(imu_100(imu[:, :, s]) * present[:, s:s + 1])
    cols.append(mag_corr_4(imu, present))
    cols.append(present)
    return np.concatenate(cols, axis=1)


def video_fingerprint(clips: np.ndarray) -> np.ndarray:
    """(N,15,768) -> L2-normalized mean∥std (N,1536). Join key, not a class feature."""
    v = np.nan_to_num(np.asarray(clips, dtype=np.float32), nan=0.0)
    emb = np.concatenate([v.mean(axis=1), v.std(axis=1)], axis=1)
    emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    return emb.astype(np.float32)


def train_lgbm(X, y, seed, colsample=0.8):
    counts = np.bincount(y, minlength=N_CLASSES).astype(np.float64) + 1.0
    sw = (1.0 / counts[y])
    sw = sw / sw.mean()
    model = lgb.LGBMClassifier(
        objective="multiclass", num_class=N_CLASSES,
        n_estimators=N_ESTIMATORS, learning_rate=LR, num_leaves=NUM_LEAVES,
        subsample=0.8, colsample_bytree=colsample, reg_lambda=1.0,
        min_child_samples=30, n_jobs=os.cpu_count() or 4,
        random_state=seed, verbose=-1,
    )
    model.fit(X, y, sample_weight=sw)
    return model


def predict_pad(model, X):
    p = model.predict_proba(X)
    if p.shape[1] < N_CLASSES:
        p = np.concatenate([p, np.zeros((len(p), N_CLASSES - p.shape[1]))], axis=1)
    return p


def viterbi(emit: np.ndarray, stay: float = STAY_PROB) -> np.ndarray:
    """emit (T, C) probabilities -> decoded class ids."""
    T, C = emit.shape
    log_stay = np.log(stay)
    log_move = np.log(max(1.0 - stay, 1e-12) / max(C - 1, 1))
    logA = np.full((C, C), log_move)
    np.fill_diagonal(logA, log_stay)
    logE = np.log(np.clip(emit, 1e-12, None))
    dp = np.empty((T, C), dtype=np.float64)
    bp = np.empty((T, C), dtype=np.int64)
    dp[0] = logE[0]
    bp[0] = -1
    for t in range(1, T):
        prev = dp[t - 1][:, None] + logA  # (C, C) from -> to
        bp[t] = prev.argmax(axis=0)
        dp[t] = prev.max(axis=0) + logE[t]
    path = np.empty(T, dtype=np.int64)
    path[-1] = int(dp[-1].argmax())
    for t in range(T - 2, -1, -1):
        path[t] = bp[t + 1, path[t + 1]]
    return path


In [ ]:
# 4. Inventory
train_csv_paths = sorted(TRAIN_INERTIAL_DIR.glob("*.csv")) or sorted(TRAIN_INERTIAL_DIR.rglob("*.csv"))
if MODE == "smoke":
    train_csv_paths = train_csv_paths[:3]
video_by_stem = {p.stem: p for p in (list(TRAIN_VIDEO_DIR.glob("*.npy")) or list(TRAIN_VIDEO_DIR.rglob("*.npy")))}
assert train_csv_paths, f"No train CSV under {TRAIN_INERTIAL_DIR}"

file_table = pd.DataFrame({
    "recording": [p.stem for p in train_csv_paths],
    "inertial_mb": [p.stat().st_size / 1024 ** 2 for p in train_csv_paths],
    "video_shape": [str(np.load(video_by_stem[p.stem], mmap_mode="r").shape) for p in train_csv_paths],
})
display(file_table.round(1).head(8))
print("n recordings", len(file_table))

test_inertial = np.load(TEST_INERTIAL_PATH, mmap_mode="r")
test_video = np.load(TEST_VIDEO_PATH, mmap_mode="r")
test_meta = pd.read_csv(TEST_META_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
target_name = "target_feature" if "target_feature" in sample_submission.columns else [c for c in sample_submission.columns if c != "id"][0]
print("test inertial", tuple(test_inertial.shape), "video", tuple(test_video.shape), "meta", test_meta.shape)
print("meta columns", list(test_meta.columns))
display(test_meta.head())
display(test_meta.groupby([c for c in ["sbj_id", "sensor_location"] if c in test_meta.columns]).size().unstack(fill_value=0) if "sensor_location" in test_meta.columns else test_meta.head())


In [ ]:
# 5. Train 4-IMU windows + sensor-dropout views
set_seed(SEED)
rows = []
for csv_path in tqdm(train_csv_paths, desc="Index recordings"):
    inertial, labels, sbj = load_inertial_csv(csv_path)
    starts, targets = segment_starts(labels, STRIDE)
    if len(starts) == 0:
        continue
    rows.append(pd.DataFrame({
        "rec": csv_path.stem, "sbj_id": sbj, "target": targets, "inertial_start": starts,
    }))
windows = subsample_subject_class(pd.concat(rows, ignore_index=True), MAX_PER_SUBJECT_CLASS, SEED)
print(f"unique 4-IMU windows={len(windows):,}")

imu_blocks, y_blocks = [], []
for rec, sub in tqdm(list(windows.groupby("rec", sort=True)), desc="Gather 4-IMU"):
    inertial, _, _ = load_inertial_csv(TRAIN_INERTIAL_DIR / f"{rec}.csv")
    starts = sub["inertial_start"].to_numpy(np.int64)
    idx = np.clip(starts[:, None] + np.arange(WINDOW_SAMPLES)[None, :], 0, inertial.shape[0] - 1)
    imu_blocks.append(inertial[idx])  # (M, 50, 4, 3)
    y_blocks.append(sub["target"].to_numpy(np.int64))
imu_w = np.concatenate(imu_blocks, axis=0)
y_w = np.concatenate(y_blocks, axis=0)
print("imu_w", imu_w.shape)

rng = np.random.RandomState(SEED)
present_list, imu_list, y_list = [], [], []
# Always keep the full 4-sensor view.
present_list.append(np.ones((len(y_w), N_SENSORS), dtype=np.float64))
imu_list.append(imu_w)
y_list.append(y_w)
# Dropout copies: test at inference has 1–4 limbs of the same second.
for _ in range(N_DROPOUT_COPIES):
    keep_n = rng.choice([1, 2, 3], size=len(y_w), p=[0.30, 0.35, 0.35])
    pres = np.zeros((len(y_w), N_SENSORS), dtype=np.float64)
    for i, k in enumerate(keep_n):
        chosen = rng.choice(N_SENSORS, size=int(k), replace=False)
        pres[i, chosen] = 1.0
    present_list.append(pres)
    imu_list.append(imu_w)
    y_list.append(y_w)

X_train = np.concatenate([features_4(imu, pres) for imu, pres in zip(imu_list, present_list)], axis=0)
y_train = np.concatenate(y_list, axis=0)
print(f"train matrix={X_train.shape} (full + dropout views)")
del imu_blocks, y_blocks, imu_list, present_list, y_list
gc.collect()


In [ ]:
# 6. Fit 4-IMU LightGBM (video is NOT a class feature)
models = []
for s in range(N_SEEDS):
    colsample = 0.75 + 0.1 * s
    print(f"training seed={SEED + s} colsample={colsample} ...", flush=True)
    models.append(train_lgbm(X_train, y_train, SEED + s, colsample=colsample))
print("trained", len(models), "boosters")


In [ ]:
# 7. Join test windows that share the same camera moment
inertial_te = np.asarray(test_inertial)
if inertial_te.ndim == 3 and inertial_te.shape[1] == 3 and inertial_te.shape[2] == WINDOW_SAMPLES:
    inertial_te = np.transpose(inertial_te, (0, 2, 1))
video_te = np.asarray(test_video)
if video_te.ndim == 3 and video_te.shape[1] == VIDEO_DIM and video_te.shape[2] == VIDEO_WINDOW:
    video_te = np.transpose(video_te, (0, 2, 1))
inertial_te = np.nan_to_num(inertial_te.astype(np.float32), nan=0.0)
video_te = np.nan_to_num(video_te.astype(np.float32), nan=0.0)

sensor_col = next(c for c in test_meta.columns if "sensor" in c.lower())
sbj_col = next(c for c in test_meta.columns if c.lower() in {"sbj_id", "subject", "subject_id"})
id_col = "id" if "id" in test_meta.columns else test_meta.columns[0]
test_ids = test_meta[id_col].to_numpy(np.int64)
sensor_ids = test_meta[sensor_col].map(SENSOR_TO_ID)
if sensor_ids.isna().any():
    raise ValueError(f"Unmapped sensors: {test_meta[sensor_col].unique()}")
sensor_ids = sensor_ids.to_numpy(np.int64)
sbj_ids = test_meta[sbj_col].to_numpy(np.int64)
n_test = len(test_ids)
print("subjects", sorted(np.unique(sbj_ids).tolist()),
      "sensors", pd.Series(sensor_ids).value_counts().sort_index().to_dict())

emb = video_fingerprint(video_te)


def summarize_groups(group_id, tag):
    stats = []
    for gid in np.unique(group_id):
        members = np.flatnonzero(group_id == gid)
        n_sens = len(set(sensor_ids[members].tolist()))
        stats.append((len(members), n_sens))
    exact4 = sum(1 for n, s in stats if s == 4 and n == 4)
    covered4 = sum(n for n, s in stats if s >= 4)
    covered2 = sum(n for n, s in stats if s >= 2)
    n3 = sum(1 for n, s in stats if s == 3)
    n2 = sum(1 for n, s in stats if s == 2)
    oversize = sum(n for n, s in stats if n > 4)
    print(
        f"[{tag}] exact4-moments={exact4}  3-sens={n3}  2-sens={n2}  "
        f"coverage4={covered4/n_test:.3f} coverage≥2={covered2/n_test:.3f} oversize_n={oversize}"
    )
    key = (exact4, covered4, n3, -oversize)
    return stats, key


def groups_from_keys(keys):
    """Same (subject, key) + different limbs → one moment. At most one window per limb."""
    group_id = np.full(n_test, -1, dtype=np.int64)
    gid = 0
    frame = pd.DataFrame({"sbj": sbj_ids, "key": keys, "sensor": sensor_ids, "idx": np.arange(n_test)})
    for (_, _), sub in frame.groupby(["sbj", "key"], sort=False):
        picked = []
        seen = set()
        for row in sub.itertuples(index=False):
            if row.sensor in seen:
                continue
            seen.add(row.sensor)
            picked.append(int(row.idx))
        for j in picked:
            group_id[j] = gid
        gid += 1
    leftover = np.flatnonzero(group_id < 0)
    for j in leftover:
        group_id[j] = gid
        gid += 1
    return group_id


def greedy_groups(thr: float):
    """1-1 matches across limb pairs, never two windows of the same limb in one group."""
    parent = np.arange(n_test)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    members = {i: {i} for i in range(n_test)}
    sensors_in = {i: {int(sensor_ids[i])} for i in range(n_test)}

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if sensors_in[ra] & sensors_in[rb]:
            return False
        if len(sensors_in[ra]) + len(sensors_in[rb]) > N_SENSORS:
            return False
        parent[rb] = ra
        members[ra] |= members[rb]
        sensors_in[ra] |= sensors_in[rb]
        return True

    for sbj in np.unique(sbj_ids):
        idx = np.flatnonzero(sbj_ids == sbj)
        for s1 in range(N_SENSORS):
            a = idx[sensor_ids[idx] == s1]
            if len(a) == 0:
                continue
            for s2 in range(s1 + 1, N_SENSORS):
                b = idx[sensor_ids[idx] == s2]
                if len(b) == 0:
                    continue
                sim = emb[a] @ emb[b].T
                order = np.argsort(sim, axis=None)[::-1]
                used_a, used_b = set(), set()
                na, nb = len(a), len(b)
                for flat in order:
                    ia, ib = divmod(int(flat), nb)
                    if sim[ia, ib] < thr:
                        break
                    if ia in used_a or ib in used_b:
                        continue
                    if union(int(a[ia]), int(b[ib])):
                        used_a.add(ia)
                        used_b.add(ib)

    roots = np.array([find(i) for i in range(n_test)])
    _, group_id = np.unique(roots, return_inverse=True)
    return group_id.astype(np.int64)


# 1) Exact VideoMAE copies (same float32 bytes) — cheapest, strongest join.
vid_keys = pd.util.hash_pandas_object(
    pd.DataFrame(video_te.reshape(n_test, -1)), index=False
).to_numpy()
gid_exact, stats_exact_key = None, None
gid_exact = groups_from_keys(vid_keys)
stats_exact, key_exact = summarize_groups(gid_exact, "exact-video-hash")

# 2) Rounded fingerprint hash (near-duplicates that are not bit-identical).
fp_keys = pd.util.hash_pandas_object(
    pd.DataFrame(np.round(emb, 5)), index=False
).to_numpy()
gid_round = groups_from_keys(fp_keys)
stats_round, key_round = summarize_groups(gid_round, "round-fp-5")

# 3) Greedy 1-1 cosine across limb pairs (no chain-merge of a whole activity).
best_g = None
best_g_key = None
best_g_thr = None
for thr in MATCH_THRESHOLDS:
    gid = greedy_groups(thr)
    stats, key = summarize_groups(gid, f"greedy@{thr}")
    if best_g_key is None or key > best_g_key:
        best_g_key, best_g, best_g_thr = key, gid, thr

candidates = [
    ("exact", gid_exact, key_exact, stats_exact),
    ("round", gid_round, key_round, stats_round),
    (f"greedy@{best_g_thr}", best_g, best_g_key, None),
]
tag, group_id, best_key, group_stats = max(candidates, key=lambda t: t[2])
if group_stats is None:
    group_stats, _ = summarize_groups(group_id, f"selected {tag}")
print(f"SELECTED JOIN={tag}  key={best_key}")
if best_key[0] < 200:
    print("WARNING: few exact 4-limb moments. 0.80 needs the join to recover ~3000 groups of 4.")


In [ ]:
# 8. Rebuild 4-IMU tensors, predict, Viterbi per subject, write submission
imu4 = np.zeros((n_test, WINDOW_SAMPLES, N_SENSORS, 3), dtype=np.float32)
present = np.zeros((n_test, N_SENSORS), dtype=np.float64)
for s in range(N_SENSORS):
    m = sensor_ids == s
    imu4[m, :, s, :] = inertial_te[m]
    present[m, s] = 1.0

# Broadcast limbs across a video-matched group (same second, other sensors).
# If a group over-merged, keep at most one window per limb — closest to the
# group's mean video fingerprint.
for gid in tqdm(np.unique(group_id), desc="Assemble moments"):
    members = np.flatnonzero(group_id == gid)
    if len(members) < 2:
        continue
    centroid = emb[members].mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
    sim_c = emb[members] @ centroid
    moment = np.zeros((WINDOW_SAMPLES, N_SENSORS, 3), dtype=np.float32)
    mask = np.zeros(N_SENSORS, dtype=np.float64)
    chosen = set()
    for rank in np.argsort(-sim_c):
        j = int(members[int(rank)])
        s = int(sensor_ids[j])
        if s in chosen:
            continue
        chosen.add(s)
        moment[:, s] = inertial_te[j]
        mask[s] = 1.0
    for j in members:
        imu4[j] = moment
        present[j] = mask

X_test = features_4(imu4, present)
print("X_test", X_test.shape, "mean n_sensors", present.sum(axis=1).mean())

probs = np.mean([predict_pad(m, X_test) for m in models], axis=0)
probs = np.clip(probs, 1e-12, None)
probs = probs / probs.sum(axis=1, keepdims=True)
probs[:, 0] *= np.exp(NULL_BIAS)
probs = probs / probs.sum(axis=1, keepdims=True)

# Average probabilities inside a moment (all limbs must agree).
for gid in np.unique(group_id):
    members = np.flatnonzero(group_id == gid)
    if len(members) < 2:
        continue
    mean_p = probs[members].mean(axis=0)
    probs[members] = mean_p

# Viterbi on the unique-moment timeline of each subject (order = min id in the group).
decoded = probs.argmax(axis=1)
for sbj in np.unique(sbj_ids):
    idx = np.flatnonzero(sbj_ids == sbj)
    gids = pd.unique(group_id[idx])
    order = []
    for gid in gids:
        members = idx[group_id[idx] == gid]
        order.append((int(test_ids[members].min()), gid, members))
    order.sort(key=lambda t: t[0])
    emit = np.stack([probs[members].mean(axis=0) for _, _, members in order], axis=0)
    path = viterbi(emit, stay=STAY_PROB)
    for c_id, (_, _, members) in zip(path, order):
        decoded[members] = c_id

pred_map = pd.Series(decoded, index=test_ids)
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
submission[target_name] = submission["id"].map(pred_map).astype(int)
assert submission[target_name].between(0, N_CLASSES - 1).all()
assert submission[target_name].notna().all()
assert len(submission) == len(sample_submission)

# Also write the unsmoothed argmax as a fallback file.
raw = pd.read_csv(SAMPLE_SUBMISSION_PATH)
raw[target_name] = raw["id"].map(pd.Series(probs.argmax(axis=1), index=test_ids)).astype(int)
raw.to_csv(OUT_DIR / "submission_nosmooth.csv", index=False)

out_path = OUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"wrote {out_path}")
display(
    submission[target_name].value_counts().rename_axis("class_id").to_frame("n")
    .reindex(range(N_CLASSES), fill_value=0).assign(activity=CLASS_NAMES)
)
display(submission.head(12))
print("group size vs sensors (selected thr):")
print(pd.DataFrame(group_stats, columns=["n_windows", "n_sensors"]).value_counts().head(12))
